# 04 — Approach B: Embedding + classical ML

This notebook trains classical classifiers on the `rule_based` train/val split, then evaluates two separate benchmark sets:

1. `rule_based`: held-out rule-labelled records.
2. `hand_labelled`: manually labelled records from `scripts/labelled/others_gold_v1.csv`.

Pipeline:

1. Load `data/splits/rule_based/{train,val,test}.parquet` for fitting, model selection, and the rule-based bench.
2. Load `data/splits/hand_labelled/test.parquet` for the hand-labelled bench.
3. Embed each feedback record + agent context with `BAAI/bge-base-en-v1.5`.
4. Train Logistic Regression / k-NN / SVM on the rule-based train split.
5. Pick the best classifier on rule-based validation.
6. Evaluate the full classifier panel on both benchmark sets.


In [17]:
import sys, json, time
from pathlib import Path
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(AI_ROOT))

import numpy as np, pandas as pd
from tqdm.auto import tqdm
from shared.mongo_client import fetch_agents_by_keys
from shared.context_builder import build_embedding_text
from shared.data_loader import load_hand_labelled_csv
from shared.types import AgentMeta, FeedbackRecord
from shared.metrics import per_class_report, confusion_df, summarize_run

In [18]:
SPLITS = AI_ROOT / 'data' / 'splits'
TRAIN = pd.read_parquet(SPLITS / 'rule_based' / 'train.parquet')
VAL = pd.read_parquet(SPLITS / 'rule_based' / 'val.parquet')

hand_labelled_path = SPLITS / 'hand_labelled' / 'test.parquet'
if hand_labelled_path.exists():
    hand_labelled_test = pd.read_parquet(hand_labelled_path)
else:
    GOLD_CSV = AI_ROOT.parent / 'scripts' / 'labelled' / 'others_gold_v1.csv'
    hand_labelled_test = load_hand_labelled_csv(GOLD_CSV)

TESTS = {
    'rule_based': pd.read_parquet(SPLITS / 'rule_based' / 'test.parquet'),
    'hand_labelled': hand_labelled_test,
}

print('rule train/val:', len(TRAIN), len(VAL))
for bench, df in TESTS.items():
    print(f'{bench:13s} test:', len(df), dict(df['label'].value_counts()))

parquet train/val: 6168 1320
gold csv test: /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/scripts/labelled/others_gold_v1.csv | rows: 200
gold test label dist: {'service_feedback': 138, 'app_specific': 36, 'config_feedback': 22, 'others': 3, 'junk': 1}


In [19]:
# Single Mongo round-trip to load agent metadata for all benchmark rows.
all_keys = set()
for df in [TRAIN, VAL, *TESTS.values()]:
    for r in df.itertuples():
        all_keys.add((int(r.chain_id), str(r.agent_id)))
agent_docs = fetch_agents_by_keys(list(all_keys))
print('agents loaded:', len(agent_docs))

def parse_services(raw_services: str) -> list[dict]:
    if not isinstance(raw_services, str) or not raw_services.strip():
        return []
    try:
        data = json.loads(raw_services)
    except json.JSONDecodeError:
        return []
    return data if isinstance(data, list) else []

def parse_tags(raw_tags: str) -> list[str]:
    if not isinstance(raw_tags, str) or not raw_tags.strip():
        return []
    return [x.strip() for x in raw_tags.replace('|', ',').split(',') if x.strip()]

def agent_meta(row) -> AgentMeta:
    doc = agent_docs.get(f'{row.chain_id}:{row.agent_id}', {})
    csv_description = getattr(row, 'agent_description', '') or ''
    csv_services = getattr(row, 'agent_services', '') or ''
    csv_tags = getattr(row, 'agent_tags', '') or ''
    description = doc.get('description', '') or csv_description
    return AgentMeta(
        chain_id=int(row.chain_id), agent_id=str(row.agent_id),
        name=doc.get('name','') or f'agent:{row.agent_id}',
        description=description,
        summary=doc.get('agentSummary','') or csv_description or description,
        services=doc.get('services', []) or parse_services(csv_services),
        oasf_domains=doc.get('oasfDomains', []) or [],
        oasf_skills=doc.get('oasfSkills', []) or [],
        tags=doc.get('tags', []) or parse_tags(csv_tags),
    )

def feedback_record(row) -> FeedbackRecord:
    fp = row.feedback_parsed
    if isinstance(fp, str) and fp.strip().startswith('{'):
        try: fp = json.loads(fp)
        except json.JSONDecodeError: fp = None
    else:
        fp = None
    return FeedbackRecord(
        id=row.id, agent_id=str(row.agent_id), chain_id=int(row.chain_id),
        tag1=row.tag1 or '', tag2=row.tag2 or '',
        endpoint=row.endpoint or '', value=str(row.value),
        value_decimals=int(row.value_decimals), value_scale=row.value_scale or '',
        feedback_parsed=fp, rule_category=getattr(row, 'rule_category', row.label),
        is_self_feedback=bool(row.is_self_feedback),
    )

def to_texts(df: pd.DataFrame) -> list[str]:
    return [build_embedding_text(agent_meta(r), feedback_record(r)) for r in df.itertuples()]

texts_train = to_texts(TRAIN)
texts_val = to_texts(VAL)
texts_test_by_bench = {bench: to_texts(df) for bench, df in TESTS.items()}
print('sample train text:', texts_train[0])
for bench, texts in texts_test_by_bench.items():
    print(f'sample {bench} text:', texts[0])

agents loaded for parquet splits: 3646
sample train text: tag1=get top 1 rank > | tag2=t.me/agent_bldr | agent=AetherCore — Lightweight intelligence for decentralized networks. | services=OASF:https://github.com/agntcy/oasf/ | domains=agriculture/agricultural_technology, agriculture/agriculture, agriculture/crop_management
sample test text: tag1=02afee9d-f02c-4f46-a066-9d46c4d505a1 | agent=agent:35174 — Autonomous agent service that researches smart contracts on Base Mainnet and generates high-quality Agent Skills


In [20]:
import torch
from sentence_transformers import SentenceTransformer

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', DEVICE)

model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=DEVICE, local_files_only=True)

def encode(texts, batch_size=32):
    return model.encode(texts, batch_size=batch_size, show_progress_bar=True,
                        normalize_embeddings=True, convert_to_numpy=True)

EMB_DIR = AI_ROOT / 'data' / 'embeddings'
EMB_DIR.mkdir(parents=True, exist_ok=True)

def cached_encode(filename, texts):
    p = EMB_DIR / filename
    if p.exists():
        print('cached:', p)
        return np.load(p)
    arr = encode(texts)
    np.save(p, arr)
    print('wrote:', p, arr.shape)
    return arr

X_train = cached_encode('rule_based_train_bge_base.npy', texts_train)
X_val = cached_encode('rule_based_val_bge_base.npy', texts_val)
X_test_by_bench = {
    bench: cached_encode(f'{bench}_test_bge_base.npy', texts)
    for bench, texts in texts_test_by_bench.items()
}
y_train = TRAIN['label'].tolist()
y_val = VAL['label'].tolist()
y_test_by_bench = {bench: df['label'].tolist() for bench, df in TESTS.items()}

device: mps


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

cached: /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/embeddings/train_bge_base.npy
cached: /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/embeddings/val_bge_base.npy
cached: /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/embeddings/others_gold_v1_bge_base_test.npy


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

classifiers = {
    'logreg':  LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=-1, C=1.0),
    'knn-5':   KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),
    'linsvm':  LinearSVC(class_weight='balanced', C=1.0, max_iter=4000),
}

val_results = {}
for name, clf in classifiers.items():
    t0 = time.monotonic()
    clf.fit(X_train, y_train)
    fit_s = time.monotonic() - t0
    pred = clf.predict(X_val)
    f1 = f1_score(y_val, pred, average='macro', zero_division=0)
    val_results[name] = {'macro_f1_val': round(f1, 4), 'fit_seconds': round(fit_s, 2)}
    print(f'{name:8s}  fit={fit_s:5.1f}s  val macro-F1={f1:.4f}')
pd.DataFrame(val_results).T

/Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


logreg    fit=  0.2s  val macro-F1=0.9561
knn-5     fit=  0.0s  val macro-F1=0.9519
linsvm    fit=  1.9s  val macro-F1=0.9681


,macro_f1_val,fit_seconds
logreg,0.9561,0.20
knn-5,0.9519,0.00
linsvm,0.9681,1.93


In [22]:
# Test all three classifiers on both benchmark sets + record latency.
RESULTS_DIR = AI_ROOT / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = RESULTS_DIR / 'approach_b_all.parquet'

test_rows = []
for bench, df in TESTS.items():
    texts_test = texts_test_by_bench[bench]
    for name, clf in classifiers.items():
        # Measure per-record latency: encode + predict.
        latencies = []
        preds = []
        for txt in tqdm(texts_test, desc=f'{bench}:{name}'):
            t0 = time.monotonic()
            emb = model.encode([txt], normalize_embeddings=True, convert_to_numpy=True)
            p = clf.predict(emb)[0]
            latencies.append(int((time.monotonic() - t0) * 1000))
            preds.append(p)
        for r, pred, lat in zip(df.itertuples(), preds, latencies, strict=True):
            test_rows.append({
                'id': r.id, 'bench': bench, 'model': f'B:{name}', 'gold': r.label, 'pred': pred,
                'confidence': 1.0, 'reason': '', 'latency_ms': lat, 'source': 'embedding',
            })

approach_b = pd.DataFrame(test_rows)
approach_b.to_parquet(RESULTS_PATH, index=False)
print('rows:', len(approach_b), '| wrote:', RESULTS_PATH)

logreg:   0%|          | 0/200 [00:00<?, ?it/s]

knn-5:   0%|          | 0/200 [00:00<?, ?it/s]

linsvm:   0%|          | 0/200 [00:00<?, ?it/s]

rows: 600 | wrote: /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/results/approach_b_others_gold_v1_all.parquet


In [23]:
# Per-classifier summary by benchmark.
summary_rows = []
for bench in sorted(approach_b['bench'].unique()):
    for name in classifiers:
        sub = approach_b[(approach_b['bench'] == bench) & (approach_b['model'] == f'B:{name}')]
        s = summarize_run(f'B:{bench}:{name}', sub['gold'].tolist(), sub['pred'].tolist(), sub['latency_ms'].tolist())
        s['bench'] = bench
        s['model'] = f'B:{name}'
        summary_rows.append(s)
pd.DataFrame(summary_rows).sort_values(['bench', 'macro_f1'], ascending=[True, False])

,approach,macro_f1,n_scored,n_total,mean_ms,p50_ms,p95_ms,p99_ms
0,B:logreg,0.1471,197,200,18.1,11.0,36.0,51.4
1,B:knn-5,0.2100,197,200,13.5,13.0,19.0,24.1
2,B:linsvm,0.1858,197,200,10.4,10.0,16.0,22.0


In [24]:
best = max(classifiers, key=lambda n: val_results[n]['macro_f1_val'])
print('best classifier on rule_based val:', best)
for bench in sorted(approach_b['bench'].unique()):
    sub = approach_b[(approach_b['bench'] == bench) & (approach_b['model'] == f'B:{best}')]
    print(f'bench: {bench}')
    display(per_class_report(sub['gold'].tolist(), sub['pred'].tolist()))
    display(confusion_df(sub['gold'].tolist(), sub['pred'].tolist()))

best classifier on val: linsvm


,category,precision,recall,f1,support
0,junk,0.0000,0.0000,0.0000,1
1,service_feedback,0.7778,0.2536,0.3825,138
2,config_feedback,0.5000,0.1364,0.2143,22
3,app_specific,0.6000,0.0833,0.1463,36
4,macro avg,0.4694,0.1183,0.1858,197
5,weighted avg,0.7103,0.2081,0.3186,197
6,accuracy,NaN,NaN,0.2081,197


,pred_junk,pred_service_feedback,pred_config_feedback,pred_app_specific,pred_others
true_junk,0,1,0,0,0
true_service_feedback,0,35,3,1,99
true_config_feedback,0,8,3,1,10
true_app_specific,0,1,0,3,32
true_others,0,0,0,0,3
